# Kiki SLM — SFT Fine-tuning on Google Colab

This notebook fine-tunes **Qwen3-4B-Instruct** with QLoRA using Unsloth on a Colab GPU.

**Prerequisites (done locally):**
1. Run `python scripts/prepare_sft_chatml.py` to download & format all 10 SFT datasets
2. Upload `data/formatted/kiki_sft_chatml_train.jsonl` and `kiki_sft_chatml_eval.jsonl` to Google Drive at:
   - `My Drive/kiki-slm/data/kiki_sft_chatml_train.jsonl`
   - `My Drive/kiki-slm/data/kiki_sft_chatml_eval.jsonl`

**What this notebook does:**
- Installs Unsloth + TRL (optimized for Colab)
- Loads ChatML-formatted data from Google Drive
- Fine-tunes Qwen3-4B with QLoRA (r=32, all attention+MLP layers)
- Saves the LoRA adapter back to Google Drive

**GPU:** Works on T4 (free), L4, A100. Training ~50K examples takes ~2-4 hrs on T4.

## 1. Install Dependencies

In [ ]:
%%capture
# Install uv package manager (ultra-fast Python package installer)
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os; os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

# Install Unsloth (handles torch/transformers/trl/peft compatibility)
!uv pip install --system unsloth
# Unsloth auto-installs compatible versions of torch, transformers, trl, peft, etc.

# Additional deps for data loading
!uv pip install --system datasets omegaconf

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 2. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the kiki-slm repo to Google Drive (persistent across sessions)
REPO_DIR = "/content/drive/MyDrive/kiki-slm/repo"

import os
if os.path.exists(f"{REPO_DIR}/.git"):
    # Repo exists — pull latest without overwriting local data
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main
    print(f"Updated existing repo at {REPO_DIR}")
else:
    !git clone https://github.com/BlueSpringsAI/kiki-slm.git {REPO_DIR}
    print(f"Cloned fresh repo to {REPO_DIR}")

# Verify repo structure
!ls {REPO_DIR}/src/kiki/

In [ ]:
# ============================================================
# CONFIGURE PATHS — edit these if your Drive layout differs
# ============================================================

DRIVE_DATA_DIR = "/content/drive/MyDrive/kiki-slm/data"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/kiki-slm/adapters"

TRAIN_FILE = f"{DRIVE_DATA_DIR}/kiki_sft_chatml_train.jsonl"
EVAL_FILE = f"{DRIVE_DATA_DIR}/kiki_sft_chatml_eval.jsonl"

# Verify data files exist
import os
assert os.path.exists(TRAIN_FILE), f"Train file not found: {TRAIN_FILE}\nUpload kiki_sft_chatml_train.jsonl to Google Drive at: kiki-slm/data/"
assert os.path.exists(EVAL_FILE), f"Eval file not found: {EVAL_FILE}\nUpload kiki_sft_chatml_eval.jsonl to Google Drive at: kiki-slm/data/"

train_size = os.path.getsize(TRAIN_FILE) / 1024 / 1024
eval_size = os.path.getsize(EVAL_FILE) / 1024 / 1024
print(f"Train file: {train_size:.1f} MB")
print(f"Eval file:  {eval_size:.1f} MB")

## 3. Load & Inspect Data

In [ ]:
from datasets import load_dataset

# Load the ChatML-formatted JSONL files
train_dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")
eval_dataset = load_dataset("json", data_files=EVAL_FILE, split="train")

print(f"Train examples: {len(train_dataset):,}")
print(f"Eval examples:  {len(eval_dataset):,}")
print(f"Columns: {train_dataset.column_names}")

In [ ]:
import json
from collections import Counter

# Show source distribution
if "source" in train_dataset.column_names:
    source_counts = Counter(train_dataset["source"])
    print("Dataset mix:")
    for source, count in sorted(source_counts.items(), key=lambda x: -x[1]):
        print(f"  {source:<30s} {count:>6,} ({count/len(train_dataset)*100:.1f}%)")

# Show a sample
print("\n" + "="*60)
print("Sample training example:")
print("="*60)
sample = train_dataset[0]
for msg in sample["messages"]:
    role = msg["role"]
    content = msg["content"][:200] + "..." if len(msg["content"]) > 200 else msg["content"]
    print(f"\n[{role}]:\n{content}")

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

# ============================================================
# MODEL CONFIGURATION
# ============================================================
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True       # QLoRA 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect (bfloat16 on A100, float16 on T4)
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters() / 1e9:.1f}B")
print(f"Quantization: {'4-bit' if LOAD_IN_4BIT else 'full precision'}")
print(f"Max seq length: {MAX_SEQ_LENGTH}")

## 5. Apply LoRA Adapters

In [ ]:
# ============================================================
# LORA CONFIGURATION
# ============================================================
LORA_R = 32                 # LoRA rank
LORA_ALPHA = 64             # LoRA alpha (2x rank is good default)
LORA_DROPOUT = 0            # Unsloth optimized — keep 0
TARGET_MODULES = [          # All attention + MLP layers for full coverage
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimized — 60% less VRAM
    random_state=42,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({trainable/total*100:.2f}%)")

## 6. Configure Training

In [ ]:
from trl import SFTTrainer, SFTConfig

# ============================================================
# TRAINING HYPERPARAMETERS
# Adjust batch_size/grad_accum based on your GPU:
#   T4 (16GB):  batch=2, grad_accum=8  → effective batch=16
#   L4 (24GB):  batch=4, grad_accum=4  → effective batch=16
#   A100 (40GB): batch=4, grad_accum=8  → effective batch=32
# ============================================================

# Auto-detect GPU and set batch size
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3

if gpu_mem_gb >= 35:  # A100
    BATCH_SIZE = 4
    GRAD_ACCUM = 8
elif gpu_mem_gb >= 20:  # L4
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
else:  # T4 or smaller
    BATCH_SIZE = 2
    GRAD_ACCUM = 8

print(f"GPU: {gpu_name} ({gpu_mem_gb:.0f}GB)")
print(f"Batch size: {BATCH_SIZE} x {GRAD_ACCUM} grad_accum = {BATCH_SIZE * GRAD_ACCUM} effective")

OUTPUT_DIR = "/content/kiki-sft-output"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    # Use our pre-formatted text column — bypasses Jinja2 entirely
    dataset_text_field="text",
    # Packing: concatenates examples to fill max_seq_length with no padding waste.
    # ~2x faster training. Loss is on all tokens (system+user+assistant).
    # This is standard practice and what Unsloth recommends.
    packing=True,
    # Batch
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    # Schedule
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    # Precision
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    # Optimizer
    optim="adamw_8bit",
    weight_decay=0.01,
    # Sequence
    max_seq_length=MAX_SEQ_LENGTH,
    # Checkpointing
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Logging & saving
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=500,
    # Misc
    report_to="none",
    seed=42,
    dataloader_pin_memory=True,
)

print(f"\nTraining config:")
print(f"  Epochs:         {training_args.num_train_epochs}")
print(f"  Learning rate:  {training_args.learning_rate}")
print(f"  Scheduler:      {training_args.lr_scheduler_type}")
print(f"  Packing:        {training_args.packing}")
print(f"  Text field:     {training_args.dataset_text_field}")
print(f"  Max seq length: {training_args.max_seq_length}")
print(f"  Precision:      {'bf16' if training_args.bf16 else 'fp16'}")

In [ ]:
# Ensure tokenizer is correctly configured for Qwen3
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Remove the 'source' column (SFTTrainer doesn't need it)
if "source" in train_dataset.column_names:
    train_dataset = train_dataset.remove_columns(["source"])
if "source" in eval_dataset.column_names:
    eval_dataset = eval_dataset.remove_columns(["source"])

# ============================================================
# PRE-FORMAT: Convert messages → plain text using the chat template.
# We do this ourselves to avoid Qwen3's Jinja2 template choking
# on the mixed dataset formats (xlam, hermes, arcee have non-standard
# message structures that the template can't handle).
# ============================================================

# Save the original chat template — we'll need it for inference later
ORIGINAL_CHAT_TEMPLATE = tokenizer.chat_template

def apply_template(examples):
    texts = []
    for msgs in examples["messages"]:
        # Sanitize: keep ONLY role + content as strings.
        # This strips tool_calls, tools, name, etc. that break the template.
        clean_msgs = []
        for m in msgs:
            clean_msgs.append({
                "role": str(m.get("role", "user")),
                "content": str(m.get("content") or ""),
            })
        try:
            text = tokenizer.apply_chat_template(
                clean_msgs, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            # Manual ChatML fallback if template still fails
            parts = []
            for m in clean_msgs:
                parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
            text = "\n".join(parts)
        texts.append(text)
    return {"text": texts}

print("Applying chat template to train set...")
train_dataset = train_dataset.map(
    apply_template, batched=True, remove_columns=["messages"],
    desc="Formatting train", num_proc=1,
)
print("Applying chat template to eval set...")
eval_dataset = eval_dataset.map(
    apply_template, batched=True, remove_columns=["messages"],
    desc="Formatting eval", num_proc=1,
)

# Verify
print(f"\nTrain: {len(train_dataset):,} examples, columns={train_dataset.column_names}")
print(f"Eval:  {len(eval_dataset):,} examples, columns={eval_dataset.column_names}")
print(f"\nSample (first 400 chars):\n{train_dataset[0]['text'][:400]}...")

# Identify the assistant response marker for loss masking
# Qwen3 ChatML format: <|im_start|>assistant\n...response...<|im_end|>
RESPONSE_MARKER = "<|im_start|>assistant\n"
print(f"\nResponse marker for loss masking: {repr(RESPONSE_MARKER)}")
assert RESPONSE_MARKER in train_dataset[0]["text"], "Response marker not found in formatted text!"
print("Response marker verified in sample.")

In [ ]:
# Disable chat template so SFTTrainer cannot invoke Jinja2.
# Our data is already pre-formatted text in the 'text' column.
tokenizer.chat_template = None

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

# Print training plan
total_steps = len(trainer.get_train_dataloader()) * training_args.num_train_epochs
print(f"\nTraining plan:")
print(f"  Total steps:     ~{total_steps:,}")
print(f"  Save every:      {training_args.save_steps} steps")
print(f"  Eval every:      {training_args.eval_steps} steps")

## 7. Train!

In [ ]:
import time

# Check GPU memory before training
print(f"GPU memory before training:")
print(f"  Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"  Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print()

start_time = time.time()
result = trainer.train()
elapsed = time.time() - start_time

print(f"\n{'='*60}")
print(f"Training complete!")
print(f"{'='*60}")
print(f"  Duration:    {elapsed/3600:.1f} hours ({elapsed/60:.0f} minutes)")
print(f"  Final loss:  {result.metrics.get('train_loss', 'N/A'):.4f}")
print(f"  Peak VRAM:   {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

In [ ]:
# Run evaluation
eval_results = trainer.evaluate()
print("\nEval results:")
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

## 8. Save Adapter to Google Drive

In [ ]:
import os, json

# Save adapter locally first
LOCAL_ADAPTER_DIR = "/content/kiki-sft-adapter"
model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)

# Copy to Google Drive
DRIVE_ADAPTER_DIR = f"{DRIVE_OUTPUT_DIR}/kiki-sft-v1"
os.makedirs(DRIVE_ADAPTER_DIR, exist_ok=True)

!cp -r {LOCAL_ADAPTER_DIR}/* {DRIVE_ADAPTER_DIR}/

print(f"\nAdapter saved to Google Drive:")
print(f"  {DRIVE_ADAPTER_DIR}")
!ls -lh {DRIVE_ADAPTER_DIR}/

# Save training metrics
metrics = {
    "model": MODEL_NAME,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "train_examples": len(train_dataset),
    "eval_examples": len(eval_dataset),
    "epochs": int(training_args.num_train_epochs),
    "learning_rate": training_args.learning_rate,
    "final_loss": result.metrics.get("train_loss"),
    "eval_results": eval_results,
    "duration_hours": round(elapsed / 3600, 2),
    "gpu": gpu_name,
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
}
with open(f"{DRIVE_ADAPTER_DIR}/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\nTraining metrics saved.")

## 9. Quick Inference Test

In [ ]:
# Restore the chat template for inference (we disabled it for training)
tokenizer.chat_template = ORIGINAL_CHAT_TEMPLATE

# Switch model to inference mode
FastLanguageModel.for_inference(model)

# Test examples
test_messages = [
    "My order #ORD-48293 hasn't arrived yet. I placed it 5 days ago.",
    "I received a damaged laptop. The screen is cracked. I want a full refund.",
    "Someone made unauthorized purchases on my account!",
]

system_prompt = """You are Kiki, an AI customer service agent. Analyze the customer message and respond with valid JSON containing: intent, urgency, workflow_steps, tools_required, reasoning, response."""

for msg in test_messages:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": msg},
    ]

    # apply_chat_template -> text, then tokenize separately
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
        )

    # Decode only the generated part
    generated = outputs[0][input_len:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"Customer: {msg}")
    print(f"-"*60)
    print(f"Kiki: {response[:500]}")

## 10. (Optional) Export GGUF for Local Inference

Export to GGUF Q4 format for running locally with llama.cpp or MLX.

In [ ]:
# Uncomment to export GGUF (takes ~5-10 min, ~3GB output)

# GGUF_OUTPUT = f"{DRIVE_OUTPUT_DIR}/kiki-sft-v1-gguf"
# os.makedirs(GGUF_OUTPUT, exist_ok=True)
#
# model.save_pretrained_gguf(
#     GGUF_OUTPUT,
#     tokenizer,
#     quantization_method="q4_k_m",
# )
# print(f"GGUF exported to: {GGUF_OUTPUT}")
# !ls -lh {GGUF_OUTPUT}/

## Done!

Your adapter is saved at: `My Drive/kiki-slm/adapters/kiki-sft-v1/`

**Next steps (locally):**
1. Download the adapter from Google Drive
2. Place it at `outputs/adapters/kiki-sft-v1/` in your project
3. Evaluate: `python scripts/3_evaluate.py --model outputs/adapters/kiki-sft-v1 --test-set data/gold/gold_100.jsonl`
4. Demo: `python scripts/4_demo.py --model outputs/adapters/kiki-sft-v1`